# Laboratorio 9 - MM3014 Teoría de Probabilidades

**Curso:** MM3014 Teoría de Probabilidades

**Nombre y Apellido:**
- Angel Sanabria (24725)
- Derek Coronado (24732)

**Fecha:** Mayo 2026

---

## Etapa 5: Simulación del Álbum Real — Mundial 2026

**Objetivo:** Aplicar los métodos de simulación desarrollados en etapas anteriores al caso real del álbum del Mundial 2026. Se formulan y responden 5 preguntas originales mediante Monte Carlo.

**Parámetros globales:**
- N = 980 estampas diferentes
- S = 7 estampas por sobre
- Precio sobre individual: Q 9.50
- Precio caja (104 sobres): Q 975.00
- Semilla: 2026
- R reducido respecto a etapas anteriores (N=980 es más costoso computacionalmente)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Configuración global
plt.style.use('seaborn-v0_8-darkgrid')

# Parámetros del álbum real
N           = 980
S           = 7
PRECIO      = 9.50
PRECIO_CAJA = 975.0
SOBRES_CAJA = 104
SEED        = 2026

In [ ]:
# -------------------------------------------------------
# Funciones auxiliares optimizadas con arrays booleanos
# numpy (mucho más rápidas que sets para N=980)
# -------------------------------------------------------

def sim_completar(S_val=None):
    """Simula hasta completar el álbum. Retorna número de sobres."""
    sv    = S_val if S_val is not None else S
    count = 0
    col   = np.zeros(N, dtype=bool)
    s     = 0
    while count < N:
        st     = np.random.choice(N, size=sv, replace=False)
        nuevas = ~col[st]
        col[st] = True
        count  += int(nuevas.sum())
        s      += 1
    return s


def sim_presupuesto(budget):
    """Simula con presupuesto fijo. Retorna True si completó el álbum."""
    count = 0
    col   = np.zeros(N, dtype=bool)
    gasto = 0.0
    while gasto + PRECIO <= budget and count < N:
        st     = np.random.choice(N, size=S, replace=False)
        nuevas = ~col[st]
        col[st] = True
        count  += int(nuevas.sum())
        gasto  += PRECIO
    return count == N


def sim_exactos(total_sobres):
    """Compra exactamente total_sobres. Retorna True si completó."""
    count = 0
    col   = np.zeros(N, dtype=bool)
    for _ in range(total_sobres):
        if count == N:
            break
        st     = np.random.choice(N, size=S, replace=False)
        nuevas = ~col[st]
        col[st] = True
        count  += int(nuevas.sum())
    return count == N


def sim_intercambio(K):
    """
    Simula hasta completar con intercambio K:1.
    Cada K repetidas acumuladas se canjean por 1 estampa faltante.
    Retorna número de sobres usados.
    """
    col       = np.zeros(N, dtype=bool)
    count     = 0
    repetidas = 0
    sobres    = 0
    while count < N:
        st     = np.random.choice(N, size=S, replace=False)
        nuevas = ~col[st]
        col[st] = True
        n_new  = int(nuevas.sum())
        count     += n_new
        repetidas += S - n_new
        sobres    += 1
        while repetidas >= K and count < N:
            falt = np.where(~col)[0]
            if len(falt) == 0:
                break
            nueva = int(np.random.choice(falt))
            col[nueva] = True
            count     += 1
            repetidas -= K
    return sobres

print('Funciones auxiliares definidas correctamente.')

---
## Pregunta 1

**¿Cuántos sobres y cuánto dinero se necesitan en promedio para completar el álbum del Mundial 2026 (N=980, S=7)?**

Se obtiene la distribución completa del número de sobres necesarios y se compara con el valor esperado teórico del Coupon Collector generalizado: $E[T] = \frac{N}{S} \cdot H_N$, donde $H_N = \sum_{k=1}^{N} \frac{1}{k}$.

In [ ]:
R_P1 = 500
np.random.seed(SEED)

sobres_p1 = np.array([sim_completar() for _ in range(R_P1)])

media_p1  = sobres_p1.mean()
std_p1    = sobres_p1.std()
H_N       = sum(1/k for k in range(1, N + 1))
E_teo     = (N / S) * H_N

print('=' * 60)
print('RESULTADOS PREGUNTA 1')
print('=' * 60)
print(f'  R = {R_P1}')
print(f'  E[sobres] simulación : {media_p1:.2f}  (std = {std_p1:.2f})')
print(f'  E[T] teórico         : {E_teo:.2f}')
print(f'  Error relativo       : {abs(media_p1 - E_teo)/E_teo*100:.2f}%')
print(f'  Costo esperado       : Q{media_p1 * PRECIO:,.2f}')
print(f'  Percentil 25         : {int(np.percentile(sobres_p1, 25))} sobres')
print(f'  Mediana              : {int(np.percentile(sobres_p1, 50))} sobres  ->  Q{int(np.percentile(sobres_p1, 50))*PRECIO:,.0f}')
print(f'  Percentil 90         : {int(np.percentile(sobres_p1, 90))} sobres  ->  Q{int(np.percentile(sobres_p1, 90))*PRECIO:,.0f}')
print('=' * 60)

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(sobres_p1, bins=40, edgecolor='white', alpha=0.82, color='steelblue')
plt.axvline(media_p1, color='crimson', linestyle='-', linewidth=2.2,
            label=f'Media = {media_p1:.0f} sobres  (Q{media_p1*PRECIO:,.0f})')
plt.axvline(E_teo, color='forestgreen', linestyle='--', linewidth=2.2,
            label=f'E[T] teórico = {E_teo:.0f}')
plt.xlabel('Sobres necesarios para completar el álbum', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.title('P1 - Distribución de sobres para completar el álbum del Mundial 2026\n'
          f'N={N}, S={S}, R={R_P1}', fontsize=13)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('pregunta1.png', dpi=150, bbox_inches='tight')
plt.show()

### Respuesta — Pregunta 1

**En promedio se necesitan ~1 037 sobres para completar el álbum**, con un costo esperado de aproximadamente **Q 9 852**. La mediana es de ~1 000 sobres (Q 9 500), ligeramente menor que la media, lo que refleja que la distribución tiene cola derecha larga: la mayoría completa cerca de los 1 000 sobres, pero algunos casos desafortunados requieren bastante más.

El valor teórico del Coupon Collector generalizado $E[T] = \frac{980}{7} \cdot H_{980} \approx 1\,045$ sobres coincide muy bien con la simulación (error < 1%), validando el modelo. Para tener el 90% de las estampas garantizado habría que estar entre los percentiles altos, que superan Q 12 000.

---
## Pregunta 2

**¿A partir de qué presupuesto se tiene al menos 50%, 75% y 90% de probabilidad de completar el álbum real?**

Se simula $\hat{P}(\text{completar} \mid \text{presupuesto} = B)$ para presupuestos de Q 4 000 a Q 14 000 en pasos de Q 1 000, comprando sobres sueltos uno a uno hasta agotar el dinero o completar el álbum.

In [ ]:
R_P2         = 300
presupuestos = list(range(4000, 14001, 1000))   # 11 puntos
prob_p2      = []

np.random.seed(SEED + 10)

for budget in presupuestos:
    exitos = sum(int(sim_presupuesto(budget)) for _ in range(R_P2))
    prob_p2.append(exitos / R_P2)

B50 = next((b for b, p in zip(presupuestos, prob_p2) if p >= 0.50), None)
B75 = next((b for b, p in zip(presupuestos, prob_p2) if p >= 0.75), None)
B90 = next((b for b, p in zip(presupuestos, prob_p2) if p >= 0.90), None)

print('=' * 55)
print('RESULTADOS PREGUNTA 2')
print('=' * 55)
for b, p in zip(presupuestos, prob_p2):
    marca = (' <- 50%' if b == B50 else
             ' <- 75%' if b == B75 else
             ' <- 90%' if b == B90 else '')
    print(f'  Q{b:,}  ->  P = {p:.3f}  ({p*100:.1f}%){marca}')
print('=' * 55)
print(f'  P >= 50% desde: Q{B50:,}' if B50 else '  P>=50%: fuera de rango')
print(f'  P >= 75% desde: Q{B75:,}' if B75 else '  P>=75%: fuera de rango')
print(f'  P >= 90% desde: Q{B90:,}' if B90 else '  P>=90%: fuera de rango')

In [ ]:
plt.figure(figsize=(11, 6))
plt.plot([b/1000 for b in presupuestos], prob_p2,
         color='steelblue', linewidth=2.5, marker='o', markersize=7)
plt.axhline(0.50, color='crimson',     linestyle='--', linewidth=1.5, alpha=0.8, label='Umbral 50%')
plt.axhline(0.75, color='darkorange',  linestyle='--', linewidth=1.5, alpha=0.8, label='Umbral 75%')
plt.axhline(0.90, color='forestgreen', linestyle='--', linewidth=1.5, alpha=0.8, label='Umbral 90%')
plt.xlabel('Presupuesto (miles de Q)', fontsize=12)
plt.ylabel('P(completar álbum)', fontsize=12)
plt.title('P2 - Probabilidad de completar el álbum según presupuesto\n'
          f'N={N}, S={S}, R={R_P2} por punto', fontsize=13)
plt.ylim(0, 1.05)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('pregunta2.png', dpi=150, bbox_inches='tight')
plt.show()

### Respuesta — Pregunta 2

Los umbrales de presupuesto obtenidos son:

| Umbral | Presupuesto necesario |
|--------|----------------------|
| P ≥ 50% | Q 10 000 |
| P ≥ 75% | Q 11 000 |
| P ≥ 90% | Q 12 000 |

**Interpretación:** Con menos de Q 9 000 la probabilidad de completar el álbum es baja (< 33%). Se necesitan al menos **Q 10 000** para tener más del 50% de probabilidad, y **Q 12 000** para alcanzar el 90%. Esto es consistente con el costo esperado de ~Q 9 852 calculado en la Pregunta 1: la mediana del costo es ligeramente inferior a Q 10 000, lo que explica por qué ese presupuesto es suficiente para la mitad de los coleccionistas.

---
## Pregunta 3

**Con un presupuesto de Q 10 000, ¿qué combinación de cajas (104 sobres, Q 975) y sobres sueltos (Q 9.50) maximiza la probabilidad de completar el álbum real?**

Se evalúan estrategias desde 0 hasta 10 cajas, usando el presupuesto restante en sobres sueltos. La caja ofrece Q 975 / 104 = Q 9.375 por sobre (más barato que suelto), por lo que más cajas implica ligeramente más sobres totales por el mismo dinero.

In [ ]:
R_P3    = 300
PRES_P3 = 10_000.0

np.random.seed(SEED + 20)

estrategias, probs_p3, labels_p3 = [], [], []

for nc in range(0, 11):
    costo_c = nc * PRECIO_CAJA
    if costo_c > PRES_P3:
        break
    ns    = int((PRES_P3 - costo_c) // PRECIO)
    tot   = nc * SOBRES_CAJA + ns
    costo = costo_c + ns * PRECIO
    estrategias.append((nc, ns, tot, costo))
    exitos = sum(int(sim_exactos(tot)) for _ in range(R_P3))
    probs_p3.append(exitos / R_P3)
    labels_p3.append(f'{nc}C+{ns}S\n({tot})')

mejor = int(np.argmax(probs_p3))

print('=' * 62)
print('RESULTADOS PREGUNTA 3')
print('=' * 62)
print(f'  {"Cajas":>5}  {"Sueltos":>7}  {"Total":>6}  {"Costo":>8}  {"P":>7}')
print(f'  {"-"*5}  {"-"*7}  {"-"*6}  {"-"*8}  {"-"*7}')
for i, ((nc, ns, tot, cst), p) in enumerate(zip(estrategias, probs_p3)):
    m = '  <- mejor' if i == mejor else ''
    print(f'  {nc:>5}  {ns:>7}  {tot:>6}  Q{cst:>7,.0f}  {p:>7.4f}{m}')
print('=' * 62)

In [ ]:
cols_p3 = ['forestgreen' if i == mejor else 'steelblue' for i in range(len(probs_p3))]

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.bar(range(len(probs_p3)), probs_p3, color=cols_p3,
              edgecolor='white', linewidth=1.1, width=0.7)
for bar, p in zip(bars, probs_p3):
    ax.text(bar.get_x() + bar.get_width()/2, p + 0.005,
            f'{p:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')
ax.set_xticks(range(len(labels_p3)))
ax.set_xticklabels(labels_p3, fontsize=8)
ax.set_ylim(0, 1.12)
ax.set_ylabel('P(completar álbum)', fontsize=12)
ax.set_title('P3 - P(completar) según estrategia con Q10,000  (C=caja 104 sob, S=suelto)\n'
             f'N={N}, S={S}, R={R_P3}  |  Verde = estrategia óptima', fontsize=11)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('pregunta3.png', dpi=150, bbox_inches='tight')
plt.show()

### Respuesta — Pregunta 3

La estrategia óptima resultó ser **7 cajas + ~334 sobres sueltos** (≈ 1 062 sobres totales, costo ≈ Q 9 998). Las diferencias entre estrategias son pequeñas (~5-10 pp) porque el número de sobres totales varía muy poco entre ellas (rango 1 052–1 066 sobres).

**¿Por qué más cajas es ligeramente mejor?** La caja ofrece Q 975/104 = Q 9.375/sobre vs Q 9.50/sobre suelto. Esto significa que con el mismo presupuesto se pueden comprar marginalmente más sobres comprando cajas, aumentando levemente la probabilidad. Sin embargo, la ganancia es modesta porque la diferencia en sobres totales es pequeña (~14 sobres entre extremos).

---
## Pregunta 4

**¿Cuánto reduce el sistema de intercambio con tasa K=5 el número esperado de sobres y el gasto total para completar el álbum real?**

Con K=5, cada 5 estampas repetidas acumuladas se pueden canjear por 1 estampa faltante (a elección). El intercambio se aplica después de cada sobre. Se comparan las distribuciones sin intercambio y con K=5 mediante histogramas superpuestos.

In [ ]:
R_P4 = 300
K_P4 = 5

np.random.seed(SEED + 30)
sobres_sinK = np.array([sim_completar() for _ in range(R_P4)])

np.random.seed(SEED + 31)
sobres_conK = np.array([sim_intercambio(K_P4) for _ in range(R_P4)])

m_sin    = sobres_sinK.mean()
m_con    = sobres_conK.mean()
ahorro_s = m_sin - m_con
ahorro_Q = ahorro_s * PRECIO
reduccion = ahorro_s / m_sin * 100

print('=' * 60)
print('RESULTADOS PREGUNTA 4')
print('=' * 60)
print(f'  R = {R_P4}  |  K = {K_P4}')
print(f'  Sin intercambio : {m_sin:.1f} sobres  ->  Q{m_sin*PRECIO:,.0f}')
print(f'  Con K={K_P4}         : {m_con:.1f} sobres  ->  Q{m_con*PRECIO:,.0f}')
print(f'  Ahorro promedio : {ahorro_s:.1f} sobres  =  Q{ahorro_Q:,.0f}')
print(f'  Reducción       : {reduccion:.1f}%')
print('=' * 60)

In [ ]:
bins4 = np.linspace(min(sobres_sinK.min(), sobres_conK.min()),
                    max(sobres_sinK.max(), sobres_conK.max()), 45)

plt.figure(figsize=(12, 6))
plt.hist(sobres_sinK, bins=bins4, color='mediumpurple', alpha=0.65, density=True,
         label=f'Sin intercambio  (media={m_sin:.0f})')
plt.hist(sobres_conK, bins=bins4, color='tomato',       alpha=0.65, density=True,
         label=f'K={K_P4}  (media={m_con:.0f})')
plt.axvline(m_sin, color='mediumpurple', linewidth=2.2, linestyle='--')
plt.axvline(m_con, color='tomato',       linewidth=2.2, linestyle='--')
plt.xlabel('Sobres para completar el álbum', fontsize=12)
plt.ylabel('Densidad', fontsize=12)
plt.title(f'P4 - Efecto del intercambio K={K_P4} sobre el número de sobres necesarios\n'
          f'N={N}, S={S}, R={R_P4}  |  Ahorro: {ahorro_s:.0f} sobres = Q{ahorro_Q:,.0f}',
          fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('pregunta4.png', dpi=150, bbox_inches='tight')
plt.show()

### Respuesta — Pregunta 4

El intercambio con K=5 produce un **ahorro de ~773 sobres en promedio**, equivalente a **~Q 7 340** (reducción del ~73%). Este resultado es llamativamente grande: con K=5 se completa el álbum con solo ~281 sobres en lugar de ~1 053.

**¿Por qué es tan eficiente?** Con N=980 y S=7, sin intercambio la gran mayoría de sobres se compran al final para buscar las últimas pocas estampas faltantes (efecto Coupon Collector). Con K=5, cada 5 repetidas se convierten en una nueva, lo que actúa como un mecanismo que elimina el cuello de botella de las últimas estampas. La distribución con intercambio es mucho más concentrada y desplazada hacia la izquierda, reflejando menor variabilidad también (el proceso se vuelve más predecible).

---
## Pregunta 5

**¿Qué tan sensible es el número de sobres necesarios al tamaño del sobre S? Si Panini cambiara el sobre de 7 a 5, 6, 8 o 9 estampas (manteniendo N=980 fijo), ¿cómo cambiaría el costo esperado para el coleccionista?**

Se simula el número esperado de sobres para S ∈ {5, 6, 7, 8, 9}, comparando contra la referencia S=7.

In [ ]:
R_P5     = 300
S_values = [5, 6, 7, 8, 9]
medias_5 = []
stds_5   = []

np.random.seed(SEED + 40)

for Sv in S_values:
    arr = np.array([sim_completar(S_val=Sv) for _ in range(R_P5)])
    medias_5.append(arr.mean())
    stds_5.append(arr.std())

ref7 = medias_5[S_values.index(7)]

print('=' * 62)
print('RESULTADOS PREGUNTA 5')
print('=' * 62)
print(f'  {"S":>3}  {"E[sobres]":>10}  {"Std":>7}  {"Costo Q":>10}  {"vs S=7":>9}')
print(f'  {"-"*3}  {"-"*10}  {"-"*7}  {"-"*10}  {"-"*9}')
for Sv, m, d in zip(S_values, medias_5, stds_5):
    diff = 'referencia' if Sv == 7 else f'{((m-ref7)/ref7*100):+.1f}%'
    print(f'  {Sv:>3}  {m:>10.1f}  {d:>7.1f}  Q{m*PRECIO:>8,.0f}  {diff:>9}')
print('=' * 62)

In [ ]:
cols5 = ['tomato', 'darkorange', 'steelblue', 'mediumseagreen', 'mediumpurple']

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(S_values, medias_5, color=cols5, edgecolor='white', linewidth=1.1, width=0.6)
ax.errorbar(S_values, medias_5, yerr=stds_5,
            fmt='none', color='#2c3e50', capsize=6, linewidth=1.8)
for bar, m in zip(bars, medias_5):
    ax.text(bar.get_x() + bar.get_width()/2, m + max(stds_5)*0.06,
            f'{m:.0f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
ax.set_xlabel('Estampas por sobre (S)', fontsize=12)
ax.set_ylabel('Sobres esperados para completar', fontsize=12)
ax.set_title('P5 - Sobres esperados según el tamaño del sobre S  (N=980)\n'
             f'R={R_P5}  |  Barras de error = 1 desv. est.', fontsize=12)
ax.set_xticks(S_values)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('pregunta5.png', dpi=150, bbox_inches='tight')
plt.show()

### Respuesta — Pregunta 5

El número esperado de sobres es fuertemente sensible al tamaño del sobre:

| S | E[sobres] | Costo Q | vs S=7 |
|---|-----------|---------|--------|
| 5 | ≈ 1 456 | ≈ Q 13 829 | +37% más caro |
| 6 | ≈ 1 209 | ≈ Q 11 486 | +14% más caro |
| **7** | **≈ 1 066** | **≈ Q 10 125** | **referencia** |
| 8 | ≈ 908 | ≈ Q 8 625 | -15% más barato |
| 9 | ≈ 816 | ≈ Q 7 749 | -23% más barato |

**Interpretación:** La relación es aproximadamente proporcional a 1/S según la fórmula teórica $E[T] \propto \frac{N}{S} \cdot H_N$. Reducir el sobre de 7 a 5 estampas encarece el álbum un 37% (~Q 3 700 adicionales), mientras que ampliarlo a 9 estampas lo abarataría un 23% (~Q 2 376 de ahorro). Desde la perspectiva del coleccionista, sobres más grandes son siempre más convenientes.

---
## Conclusiones

1. **Costo real del álbum (P1):** Completar el álbum del Mundial 2026 cuesta en promedio **~Q 9 852**, con alta variabilidad (desv. est. ~Q 1 767). El 10% más "desafortunado" de los coleccionistas gasta más de Q 12 000.

2. **Presupuesto necesario (P2):** Para tener 50% de probabilidad de completar se necesitan **Q 10 000**; para 90%, **Q 12 000**. Con menos de Q 9 000 la probabilidad es inferior al 33%.

3. **Estrategia óptima de compra (P3):** Con Q 10 000, comprar cajas es ligeramente mejor que sólo sueltos porque el precio por sobre es menor (Q 9.375 vs Q 9.50). La diferencia es pequeña (~5-10 pp) porque la ganancia en sobres totales es marginal.

4. **Impacto del intercambio (P4):** Un sistema de intercambio con K=5 reduce el costo esperado en **~73%** (de ~Q 10 000 a ~Q 2 700). Es la palanca más poderosa identificada: elimina el cuello de botella de las últimas estampas faltantes.

5. **Sensibilidad al tamaño del sobre (P5):** Cada estampa adicional por sobre reduce el costo de forma proporcional. Pasar de S=7 a S=9 ahorraría ~Q 2 400; pasar a S=5 encarecería el álbum ~Q 3 700.